# 51. The Nesterov prefit optimizer

**Objectives:**

- Fit the same small model with `method="minuit"` (the default) and `method="nesterov"`.
- Compare the recovered parameter values between the two.
- Inspect `NesterovResult`'s Minuit-compatible fields and its `fmin.edm` limitation.

Run the cells in order in a fresh kernel. Masses are in GeV, invariants in GeV^2, and daughter
indices start at zero.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede any numerical work: amplitudes use complex128.

import numpy as np

from dalitzplotfitter import (
    DecayChannel, DecayModel, FitSession, NonResonant,
    Parameter, RealImag, Resonance, generate_toy,
)

## 1. Model and toy data

Same rho(770) + non-resonant model as the other lessons in this group, with a floatable
Cartesian coefficient on the non-resonant term.

In [2]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
x = Parameter.coefficient("NR.x", 0.55, owner="NR", bounds=(-2, 2), step=0.02)
y = Parameter.coefficient("NR.y", 0.30, owner="NR", bounds=(-2, 2), step=0.02)
components = [
    Resonance("rho", (0, 1), RealImag(1, 0), mass=0.7753, width=0.1491, spin=1),
    NonResonant(RealImag(x, y), name="NR"),
]
model = DecayModel(
    channel, components, normalization_method="square-dalitz",
    normalization_resolution=60, normalization_pair=(0, 1),
)
truth = {p.name: p.value for p in model.parameters}

data = generate_toy(
    model, 1500, parameters=truth, seed=2026,
    method="inverse-transform", inverse_resolution=256, include_momenta=False,
)
session = FitSession(model, data)
print(f"Generated {data.size} unweighted events; truth = {truth}")

Generated 1500 unweighted events; truth = {'NR.x': 0.55, 'NR.y': 0.3}


## 2. Fit with Minuit (default) and with Nesterov

`Minimizer.fit`/`FitSession.fit` accepts `method="minuit"` (default), `method="nesterov"` (a
projected accelerated-gradient prefit, returned directly as the final result) and
`method="nesterov-minuit"` (Nesterov as a starting point for the ordinary Minuit fit).
`method="nesterov"` has no Minuit dependency at all: it runs pure JAX gradient steps in
parameter-scaled, bounded coordinates.

In [3]:
start = {"NR.x": 0.35, "NR.y": 0.45}

result_minuit = session.fit(start, strategy=1, hesse=True)
result_nesterov = session.fit(start, method="nesterov", strategy=1)

print("method=minuit   :", {n: round(float(result_minuit.values[n]), 4) for n in result_minuit.parameters})
print("method=nesterov :", {n: round(v, 4) for n, v in result_nesterov.values.items()})
print(f"NLL(minuit)   = {float(result_minuit.fval):.6f}")
print(f"NLL(nesterov) = {float(result_nesterov.fval):.6f}")

method=minuit   : {'NR.x': 0.6057, 'NR.y': 0.3122}
method=nesterov : {'NR.x': 0.6057, 'NR.y': 0.3122}
NLL(minuit)   = 1854.908813
NLL(nesterov) = 1854.908813


## 3. `NesterovResult` fields

`NesterovResult` is a Minuit-compatible stand-in: `values`, `fval`, `valid`, `status`,
`converged`, `history`, `nfcn`, `errors` and `covariance` all exist so it can be handed to code
written against a Minuit result. Its `fmin` property is a `SimpleNamespace` whose `edm` is
always `nan`, because Nesterov has no EDM-based convergence check -- do not compare it to a
Minuit `fmin.edm` value.

In [4]:
print(f"optimizer   = {result_nesterov.optimizer}")
print(f"status      = {result_nesterov.status}")
print(f"converged   = {result_nesterov.converged}")
print(f"nfcn        = {result_nesterov.nfcn}")
print(f"history len = {len(result_nesterov.history)}")
print(f"fmin.edm    = {result_nesterov.fmin.edm}  (always nan -- no EDM check)")

optimizer   = nesterov
status      = converged
converged   = True
nfcn        = 129
history len = 6
fmin.edm    = nan  (always nan -- no EDM check)


## Summary and exercises

1. Try `method="nesterov-minuit"` and compare its result and `nfcn` to plain `method="minuit"`.
2. Start further from the truth and see whether the Nesterov prefit alone still lands close to
   the Minuit result.
3. Because `fmin.edm` is always `nan`, do not gate downstream code on it for a Nesterov-only
   result; use `.valid`/`.converged` instead.

Return to the [course guide](TUTORIALS.md).